In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

%matplotlib inline
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
Anonymized_Data_path = os.path.join(path, 'Q3_data.csv')
Anonymized_Data_df = pd.read_csv(Anonymized_Data_path)

In [ ]:
# Task 2: Write your code here:
Anonymized_Data_df.head()

In [ ]:
# Task 3: Write your code here:
Anonymized_Data_df.info()

In [ ]:
# Task 4: Write your code here:
Anonymized_Data_df.describe()

In [ ]:
# Task 1: Write your code here:
print("Missing values:")
print(Anonymized_Data_df.isnull().sum())
df_clean = df_clean.fillna(np.median(0))
print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 2: Write your code here:
duplicate_rows = df_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
#We dont need to encoding them


#label_encoders = {}
#for col in categorical_cols:
  #le = LabelEncoder()
  #df_clean[col] = le.fit_transform(df_clean[col])
  #label_encoders[col] = le

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df_clean[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()
check_target_imbalance(df_clean, "Target")


In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Target", axis=1).astype(float)
y = df_clean['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier

#the catboost library couldn't be running on my device due to my old colab version

sklearn_models = {
    "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'f1': []}

n_splits = 5 # K
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['f1'].append(f1)

print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")


In [ ]:
# Task 1: Write your code here:
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_
importance = df_clean.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: